In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mazedev/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mazedev/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mazedev/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mazedev/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas7_2505060012").master("local[*]").getOrCreate()

print("Spark Session berhasil dibuat: ", spark.version)

Spark Session berhasil dibuat:  3.5.9


In [6]:
# Membaca ketiga sumber data dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mazedev/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mazedev/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mazedev/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("===Schema DF_Transaksi===")
df_transaksi.printSchema()

print("\n===Schema DF_Cabang===")
df_cabang.printSchema()

print("\n===Schema DF_Menu===")
df_menu.printSchema()

===Schema DF_Transaksi===
root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)


===Schema DF_Cabang===
root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)


===Schema DF_Menu===
root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)



In [7]:
from pyspark.sql.functions import col
# Join ketiga tabel
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner").join(df_menu, on="menu_id", how="inner")

# tambahkan kolom
df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show()

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [12]:
from pyspark.sql.functions import sum as spark_sum, row_number
from pyspark.sql.window import Window

df_terlaris = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

window = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan"))

df_top2 = df_terlaris.withColumn("peringkat", row_number().over(window)).filter(col("peringkat") < 3)

df_top2.show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|  Menu-10|        3762000|        1|
|Kopi Nusantara Ma...|  Menu-14|        4050000|        2|
|Kopi Nusantara Se...|  Menu-14|        3366000|        1|
|Kopi Nusantara Se...|   Menu-7|        3456000|        2|
|Kopi Nusantara Yo...|  Menu-10|        3636000|        1|
|Kopi Nusantara Yo...|   Menu-7|        3816000|        2|
+--------------------+---------+---------------+---------+



In [13]:
df_gabungan.createOrReplaceTempView("gabungan")

spark_sql = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) as total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')

spark_sql.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [14]:
#Menyimpan ke HDFS dalam bentuk parquet
!hdfs dfs -mkdir -p /user/mazedev/tugas7/processed/laporan

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet("hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan")

!hdfs dfs -ls "hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan"

Found 4 items
-rw-r--r--   3 mazedev supergroup          0 2026-10-10 21:14 hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - mazedev supergroup          0 2026-10-10 21:14 hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - mazedev supergroup          0 2026-10-10 21:14 hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - mazedev supergroup          0 2026-10-10 21:14 hdfs://localhost:9000/user/mazedev/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [17]:
spark.stop()

print("Spark Session Stop.")

Spark Session Stop.
